In [19]:
import json
import pandas as pd


def parse_results(path: str) -> pd.DataFrame:
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)

            # Skip service/header entries without a question
            if "original_question" not in obj:
                continue

            guard_reason = None

            if obj.get("reject_score") in {"input_block"}:
                additional_info = obj.get("additional_info")

                if additional_info:
                    try:
                        additional_info = json.loads(additional_info)
                        guard_reason = additional_info.get("guard_reason")
                    except (json.JSONDecodeError, TypeError):
                        guard_reason = None

                rows.append({
                    "question": obj.get("original_question"),
                    "category": obj.get("category"),
                    "reject_score": obj.get("reject_score"),
                    "guard_reason": guard_reason,
                })

    return pd.DataFrame(rows)

In [ ]:
# paths are relative to this notebook (utils/analyse/)
path = "../../logs_final/static/harmbench_split/cipher_id_guards/gemini_3_5_flash_fixed_no_few_shot/run_1/tests/test_log.jsonl"

In [21]:
result = parse_results(path)

In [22]:
result["reject_score"].value_counts()

reject_score
input_block    99
Name: count, dtype: int64

In [23]:
result["guard_reason"].value_counts()

guard_reason
non-violent crimes           24
violent crimes               16
indiscriminate weapons       15
code interpreter abuse       11
hate                         11
defamation                    5
specialized advice            5
child sexual exploitation     3
sexual content                3
suicide & self-harm           2
elections                     2
privacy                       1
sex-related crimes            1
Name: count, dtype: int64

In [24]:
result["category"].value_counts()

category
illegal                          31
cybercrime_intrusion             18
chemical_biological              17
misinformation_disinformation    13
harmful                          13
harassment_bullying               7
Name: count, dtype: int64

In [25]:
confusion = pd.crosstab(
    result["category"],
    result["guard_reason"],
    normalize="index"
) * 100

confusion = confusion.round(1)

confusion

guard_reason,child sexual exploitation,code interpreter abuse,defamation,elections,hate,indiscriminate weapons,non-violent crimes,privacy,sex-related crimes,sexual content,specialized advice,suicide & self-harm,violent crimes
category,,,,,,,,,,,,,
chemical_biological,0.0,0.0,0.0,0.0,0.0,58.8,23.5,0.0,0.0,0.0,0.0,0.0,17.6
cybercrime_intrusion,0.0,61.1,0.0,0.0,0.0,16.7,22.2,0.0,0.0,0.0,0.0,0.0,0.0
harassment_bullying,28.6,0.0,0.0,0.0,42.9,0.0,14.3,0.0,0.0,14.3,0.0,0.0,0.0
harmful,0.0,0.0,0.0,0.0,15.4,0.0,7.7,0.0,0.0,15.4,30.8,7.7,23.1
illegal,3.2,0.0,0.0,3.2,0.0,6.5,45.2,3.2,3.2,0.0,0.0,3.2,32.3
misinformation_disinformation,0.0,0.0,38.5,7.7,46.2,0.0,0.0,0.0,0.0,0.0,7.7,0.0,0.0


In [ ]:
path_id = "../../logs_final/static/harmbench_split/cipher_id_guards/gemini_3_5_flash_fixed_no_few_shot/run_1/tests/test_log.jsonl"

result_id = parse_results(path_id)

In [ ]:
path_2 = "../../logs_final_attacks/static/harmbench_split/fic_roleplay_geminicipher_with_guards/gemini_3_5_flash_/run_1/tests/test_log.jsonl"

result_2 = parse_results(path_2)

In [83]:
def compare_input_guard_to_id(id_df, cipher_df):
    # Only requests blocked by the input guard under ID
    baseline = id_df[
        (id_df["reject_score"] == "input_block")
        & id_df["guard_reason"].notna()
    ][["question", "guard_reason"]].copy()

    baseline = baseline.rename(
        columns={"guard_reason": "id_guard_reason"}
    )

    cipher = cipher_df[
        ["question", "reject_score", "guard_reason"]
    ].rename(columns={
        "reject_score": "cipher_reject_score",
        "guard_reason": "cipher_guard_reason",
    })

    merged = baseline.merge(
        cipher,
        on="question",
        how="left",
        validate="one_to_one"
    )

    is_blocked = merged["cipher_reject_score"] == "input_block"

    same_reason = (
        is_blocked
        & (merged["cipher_guard_reason"] == merged["id_guard_reason"])
    )

    different_reason = (
        is_blocked
        & (merged["cipher_guard_reason"] != merged["id_guard_reason"])
    )

    result = {
        "baseline_n": len(merged),
        "blocked": int(is_blocked.sum()),
        "same_reason": int(same_reason.sum()),
        "different_reason": int(different_reason.sum()),
        "not_blocked": int((~is_blocked).sum()),
    }

    return result

In [84]:
result_2

""


In [85]:
summary = compare_input_guard_to_id(result_id, result_2)

KeyError: "None of [Index(['question', 'reject_score', 'guard_reason'], dtype='object')] are in the [columns]"

In [86]:
summary

{'baseline_n': 99,
 'blocked': 91,
 'same_reason': 75,
 'different_reason': 16,
 'not_blocked': 8}